In [0]:
import pyspark
from pyspark.sql import SparkSession
# from pyspark.sql.functions import year, month, dayofmonth, hour, minute, mean, median, sum, count, col, first
import pyspark.sql.functions as f

In [0]:
spark = SparkSession.builder\
    .appName("EnergyData")\
    .getOrCreate()

In [0]:
df = spark.read.parquet("s3://weave.energy/smart-meter.parquet")

In [0]:
df.printSchema()

In [0]:
df.select('aggregated_device_count_active').show()

In [0]:
df = df.withColumn('year', f.year('data_collection_log_timestamp')).\
        withColumn('month', f.month('data_collection_log_timestamp')).\
        withColumn('day', f.dayofmonth('data_collection_log_timestamp')).\
        withColumn('hour', f.hour('data_collection_log_timestamp'))

In [0]:
df.select('data_collection_log_timestamp', 'year', 'month', 'day', 'hour').show()

In [0]:
df.groupBy('year', 'month', 'day', 'hour').agg(
    f.sum(f.col('total_consumption_active_import')).alias('total_consumption'),
    f.first(f.col('aggregated_device_count_active')).alias('aggregated_device_count'),
    f.mean(f.col('aggregated_device_count_active')).alias('mean_aggregated_device_count'),
    f.sum(f.col('aggregated_device_count_active')).alias('total_devices_measured'),
    f.max(f.col('total_consumption_active_import')).alias('max_consumption'),
    f.min(f.col('total_consumption_active_import')).alias('min_consumption'),
    f.mean(f.col('total_consumption_active_import')).alias('mean_consumption')
).show()